ссылка на ноутбук
https://cloud.mail.ru/public/ZRjF/LoHQtzDZb

In [51]:
import numpy as np
import pandas as pd

# ============================================================
# 1. Конфигурация
# ============================================================

DATA_PATH = "data/drom_used_cars.csv"
TARGET_COL = "price_rub"
REFERENCE_YEAR = 2026

DROP_COLS = [
    "listing_id",
    "source_url",
    "search_url",
    "model_rank",
    "model_id",
    "collected_at_utc",
    "price_type",
    "condition",
    "labels",
    "trim",
    "year",
]

TRANSMISSION_MAPPING = {
    "робот": "автомат",
    "АКПП": "автомат",
    "вариатор": "автомат",
}

# ============================================================
# 2. Загрузка данных
# ============================================================

df = pd.read_csv(DATA_PATH)

print(f"Исходный размер датасета: {df.shape}")

# ============================================================
# 3. Создание и преобразование признаков
# ============================================================

# Переводим стоимость автомобиля в миллионы рублей
df[TARGET_COL] = df[TARGET_COL] / 1_000_000

# Рассчитываем возраст автомобиля
df["age"] = REFERENCE_YEAR - df["year"]

# Объединяем типы автоматических трансмиссий
df["transmission_type"] = df["transmission"].replace(
    TRANSMISSION_MAPPING
)

# ============================================================
# 4. Очистка данных
# ============================================================

# Исключаем автомобили с некорректным объемом двигателя
df = df.loc[df["engine_volume_l"] < 6.5].copy()

# Удаляем ненужные столбцы и строки с пропусками
df = df.drop(columns=DROP_COLS).dropna()

print(f"Размер датасета после очистки: {df.shape}")

# ============================================================
# 5. Определение типов признаков
# ============================================================

num_cols = (
    df.select_dtypes(include="number")
    .columns.drop(TARGET_COL)
    .tolist()
)

cat_cols = (
    df.select_dtypes(exclude="number")
    .columns.tolist()
)

print(f"\nЧисловые признаки: {num_cols}")
print(f"Категориальные признаки: {cat_cols}")


Исходный размер датасета: (8709, 26)
Размер датасета после очистки: (8560, 17)

Числовые признаки: ['engine_volume_l', 'power_hp', 'mileage_km', 'is_damaged', 'age']
Категориальные признаки: ['brand', 'brand_group', 'brand_country', 'model', 'fuel_type', 'transmission', 'drive', 'body_type', 'city', 'attributes_raw', 'transmission_type']


## Сопоставление гиперпараметров трёх библиотек

Эту таблицу особенно полезно добавить в материалы семинара: она показывает, какие параметры отвечают за один и тот же механизм, но называются по-разному.

| Назначение | XGBoost | LightGBM | CatBoost |
| --- | --- | --- | --- |
| Число итераций | `n_estimators` | `n_estimators` | `iterations` |
| Скорость обучения | `learning_rate` | `learning_rate` | `learning_rate` |
| Глубина | `max_depth` | `max_depth` | `depth` |
| Максимум листьев | `max_leaves` | `num_leaves` | `max_leaves` (Lossguide) |
| Минимальная сумма гессианов | `min_child_weight` | `min_child_weight` | Нет прямого аналога общего назначения |
| Минимум объектов в листе | Нет прямого аналога | `min_child_samples` | `min_data_in_leaf` (Depthwise/Lossguide) |
| Минимальный gain для split | `gamma` | `min_split_gain` | Нет прямого аналога |
| L1-регуляризация листьев | `reg_alpha` | `reg_alpha` | Нет прямого аналога |
| L2-регуляризация листьев | `reg_lambda` | `reg_lambda` | l2_leaf_reg |
| Доля объектов | `subsample` | `subsample` + `subsample_freq` | `subsample` (зависит от `bootstrap`) |
| Доля признаков на дерево | `colsample_bytree` | `colsample_bytree` | Нет прямого аналога |
| Доля признаков при split | `colsample_bynode` | `feature_fraction_bynode` | `rsm` (поиск `split`) |
| Количество бинов | `max_bin` | `max_bin` | `border_count` |
| Стратегия роста | `grow_policy` | Leaf-`wise` | `grow_policy` |
| Способ работы с категориями | `enable_categorical` | `categorical_feature` | `cat_features` |
| Выбор CPU/GPU | `device` | `device_type` | `task_type` |
| Число CPU-потоков | `n_jobs` | `n_jobs` | `thread_count` |
| Seed | `random_state` | `random_state` | `random_seed` |




# XGBoost

|Параметр	|Что означает	|Как влияет|
|---|---|---|
|n_estimators|	Количество итераций (деревьев)|	Больше итераций → модель может выучить более сложную зависимость, но повышается риск переобучения|
|learning_rate|	Размер вклада очередного дерева	|Меньше → обучение медленнее, обычно требуется больше деревьев|
|booster|	Тип базовой модели: gbtree, dart, gblinear|	Определяет архитектуру ансамбля|
|objective|	Функция потерь, которую минимизирует модель	|Определяет задачу и способ вычисления градиентов|
|eval_metric|	Метрика для мониторинга качества	|Используется для оценки качества и early stopping|
|**Параметры сложности деревьев**|
|max_depth|	Максимальная глубина дерева|	Более сложные зависимости, выше риск overfitting|
|min_child_weight|	Минимальная сумма гессианов в дочернем узле	|При увеличении параметра меньше разбиений, более консервативные деревья
|gamma (min_split_loss)|	Минимальное улучшение objective для разбиения|	Меньше разбиений|
|max_leaves|	Максимальное число листьев|	Более сложные деревья|
|grow_policy|	Стратегия роста дерева depthwise/lossguide	|Меняет форму деревьев: рост по уровням, с учётом ограничения глубины/приоритетное разбиение листа с наибольшим выигрышем|
|max_bin|	Количество бинов для числовых признаков при histogram-based построении|	Более точный поиск порогов, но выше расход памяти и время|
|**Регуляризация**|
|reg_lambda|	L2-регуляризация весов листьев|	Значения листьев сильнее сжимаются|
|reg_alpha|	L1-регуляризация весов листьев|	Некоторые значения листьев могут стать нулевыми|
|gamma	|Штраф за дополнительные листья|	Деревья становятся проще|
|max_delta_step	|Ограничение абсолютного шага обновления веса листа|	Обновления становятся более осторожными|
|**Случайный выбор наблюдений и признаков**| Уменьшение корреляции между деревьями и борьба с переобучением.|
|subsample|	Доля наблюдений для очередного дерева|	0.8 — 80% объектов|
|colsample_bytree|	Доля признаков для каждого дерева|	0.8 — 80% признаков|
|colsample_bylevel|	Доля признаков на каждом уровне дерева|	0.8|
|colsample_bynode|	Доля признаков при каждом разбиении|	0.8|
|sampling_method|	Метод отбора объектов|	uniform или gradient_based|
|**Параметры построения деревьев и производительности**|
|tree_method|	auto, hist, exact, approx|	Алгоритм поиска разбиений|
|max_bin|	Целое число	|Точность гистограммного поиска порогов|
|device|	cpu, cuda	|Где выполнять вычисления|
|n_jobs	|Целое число|	Число потоков CPU|
|max_cached_hist_node|	Целое число|	Ограничение кеша гистограмм|
|random_state|	Целое число	|Воспроизводимость случайных процедур?|
  
**Параметры категориальных признаков**
|Параметр|Что означает|
|---|---|
|enable_categorical	|Разрешает нативную работу с категориальными признаками|
|max_cat_to_onehot|	Порог количества категорий для выбора one-hot-подобного разбиения вместо partition-based|
|max_cat_threshold|	Ограничивает число категорий, рассматриваемых при поиске категориального разбиения|





lossguide может быть полезен для данных с неоднородной сложностью: например, когда для большинства клиентов зависимость простая, но небольшая группа требует более детальной сегментации.

XGBRegressor(
    colsample_bytree=0.8,
    colsample_bylevel=0.5,
    colsample_bynode=0.5
)
Сначала для дерева выбирается примерно 80 признаков. Затем на уровне доступно примерно 40, а для конкретного узла — примерно 20.
Случайный выбор признаков здесь в отличие от RF выступает дополнительным способом регуляризации.


gamma
Параметр задаёт минимальный выигрыш от разбиения узла.

В XGBoost выигрыш от split при стандартной регуляризации имеет вид:

$$
\mathrm{Gain}
=
\frac{1}{2}
\left[
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{G^2}{H+\lambda}
\right]
-
\gamma
$$


Регуляризация
XGBoost использует регуляризацию непосредственно при оптимизации деревьев.

В стандартной постановке:
$$
\Omega(f)
=
\gamma T
+
\frac{\lambda}{2}\sum_{j=1}^{T} w_j^2
+
\alpha\sum_{j=1}^{T} \lvert w_j \rvert
$$
Здесь \(T\) — число листьев, а \(w_j\) — значение поправки в листе.

Как работает reg_lambda
Оптимальный вес листа без L1-регуляризации:
$$
\boxed{w_j^*=-\frac{G_j}{H_j+\lambda}}
$$

Чем больше $\lambda$, тем меньше абсолютная величина поправки.

In [33]:
from sklearn.model_selection import train_test_split

cols = num_cols + cat_cols
df[cat_cols] = df[cat_cols].astype('category')
X_train, X_test, y_train, y_test = train_test_split(df[cols], df[TARGET_COL], test_size=.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=.2, random_state=42)

In [54]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

model = xgb.XGBRegressor(early_stopping_rounds=50, n_estimatots=300)
model.fit(X_train, y_train, 
        eval_set=[(X_valid, y_valid)], 
        verbose=False)

y_pred = model.predict(X_valid)
mae = mean_absolute_error(y_valid, y_pred)
print(mae)

/Users/irinakharkova/my_projects/ml_course/2 семинар линейнай регрессия/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [10:19:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "n_estimatots" } are not used.

  self.starting_round = model.num_boosted_rounds()


0.7123754916257664


In [55]:
model.best_iteration

99

## Optuna

https://optuna.readthedocs.io/en/stable/?utm_source=chatgpt.com

Python-библиотека для автоматического подбора гиперпараметров моделей машинного обучения.

Optuna (с алгоритмом TPE) — использует результаты предыдущих экспериментов для выбора следующих комбинаций.

In [56]:

# pip install optuna xgboost

import optuna
import matplotlib.pyplot as plt


optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42

def objective(trial):

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "mae",
        "enable_categorical": True,
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,

        # Скорость обучения
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.3, log=True
        ),

        # Сложность дерева
        "max_depth": trial.suggest_int(
            "max_depth", 2, 10
        ),
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1, 20
        ),
        "gamma": trial.suggest_float(
            "gamma", 0, 10
        ),

        # Случайный выбор объектов и признаков
        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        # Регуляризация листьев
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 100, log=True
        ),

        # Число деревьев определяем через early stopping
        "n_estimators": 2000,
        "early_stopping_rounds": 50,
    }

    model = xgb.XGBRegressor(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    y_pred = model.predict(X_valid)
    mae = mean_absolute_error(y_valid, y_pred)

    trial.set_user_attr(
        "best_iteration", model.best_iteration
    )

    return mae


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

study.optimize(
    objective,
    n_trials=10,
    show_progress_bar=True
)

print("Best MAE:", study.best_value)
print("Best params:")
print(study.best_params)
print("Best iteration:",
      study.best_trial.user_attrs["best_iteration"])


  0%|          | 0/10 [00:00<?, ?it/s]

Best MAE: 0.5515418639129375
Best params:
{'learning_rate': 0.028145092716060652, 'max_depth': 6, 'min_child_weight': 9.206955354200199, 'gamma': 2.9122914019804194, 'subsample': 0.8447411578889518, 'colsample_bytree': 0.6557975442608167, 'reg_alpha': 4.258943089524393e-06, 'reg_lambda': 0.06789053271698488}
Best iteration: 558


In [57]:
best_params = study.best_params.copy()

best_n_estimators = (
    study.best_trial.user_attrs["best_iteration"] + 1
)

X_train_full = pd.concat([X_train, X_valid])
y_train_full = pd.concat([y_train, y_valid])

best_model = xgb.XGBRegressor(
    **best_params,
    n_estimators=best_n_estimators,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

best_model.fit(X_train_full, y_train_full)

y_pred_test = best_model.predict(X_test)

print("Test MAE:",
      mean_absolute_error(y_test, y_pred_test))
print("Test RMSE:",
      root_mean_squared_error(y_test, y_pred_test))

Test MAE: 0.44127875631460967
Test RMSE: 1.2673665803994163


In [58]:

import optuna.visualization as vis

# Как менялось лучшее качество
vis.plot_optimization_history(study).show()

# Важность гиперпараметров
vis.plot_param_importances(study).show()

# Зависимость качества от гиперпараметров
vis.plot_slice(study).show()

In [12]:

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42


# Native XGBoost: категории передаются как pandas category
X_train_native = X_train.copy()
X_valid_native = X_valid.copy()

for col in cat_cols:
    X_train_native[col] = X_train_native[col].astype("category")
    X_valid_native[col] = X_valid_native[col].astype("category")

# OneHotEncoder обучаем только на train
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

X_train_ohe = preprocessor.fit_transform(X_train)
X_valid_ohe = preprocessor.transform(X_valid)


def objective(trial):

    encoding = trial.suggest_categorical(
        "encoding", ["native", "onehot"]
    )

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "mae",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,

        "n_estimators": 2000,
        "early_stopping_rounds": 50,

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 2, 9
        ),
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1, 20
        ),
        "gamma": trial.suggest_float(
            "gamma", 0, 10
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 100, log=True
        )
    }

    if encoding == "native":

        params["enable_categorical"] = True

        params["max_cat_to_onehot"] = trial.suggest_int(
            "max_cat_to_onehot", 2, 10
        )

        params["max_cat_threshold"] = trial.suggest_int(
            "max_cat_threshold", 8, 64
        )

        X_tr = X_train_native
        X_val = X_valid_native

    else:
        params["enable_categorical"] = False

        X_tr = X_train_ohe
        X_val = X_valid_ohe

    model = xgb.XGBRegressor(**params)

    model.fit(
        X_tr,
        y_train,
        eval_set=[(X_val, y_valid)],
        verbose=False
    )

    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_valid, y_pred)

    trial.set_user_attr(
        "best_iteration", model.best_iteration
    )

    return mae


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

# Чтобы гарантировать хотя бы один запуск каждого варианта
study.enqueue_trial({"encoding": "native"})
study.enqueue_trial({"encoding": "onehot"})

study.optimize(
    objective,
    n_trials=10,
    show_progress_bar=True
)

print("Best MAE:", study.best_value)
print("Best encoding:", study.best_params["encoding"])
print("Best params:", study.best_params)


  0%|          | 0/10 [00:00<?, ?it/s]

Best MAE: 0.32345515031772426
Best encoding: onehot
Best params: {'encoding': 'onehot', 'learning_rate': 0.010636066512540286, 'max_depth': 9, 'min_child_weight': 16.816410175208013, 'gamma': 2.1233911067827616, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 5.472429642032189e-06, 'reg_lambda': 0.42051564509138706}


In [13]:

best_params = study.best_params.copy()

best_n_estimators = (
    study.best_trial.user_attrs["best_iteration"] + 1
)

X_train_full = pd.concat([X_train, X_valid])
y_train_full = pd.concat([y_train, y_valid])

best_model = xgb.XGBRegressor(
    **best_params,
    n_estimators=best_n_estimators,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

best_model.fit(X_train_full, y_train_full)

y_pred_test = best_model.predict(X_test)

print("Test MAE:",
      mean_absolute_error(y_test, y_pred_test))
print("Test RMSE:",
      root_mean_squared_error(y_test, y_pred_test))


/Users/irinakharkova/my_projects/ml_course/2 семинар линейнай регрессия/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:13:07] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "encoding" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Test MAE: 0.5180818077698923
Test RMSE: 1.3922563819119704


In [14]:
# Как менялось лучшее качество
vis.plot_optimization_history(study).show()

# Важность гиперпараметров
vis.plot_param_importances(study).show()

# Зависимость качества от гиперпараметров
vis.plot_slice(study).show()


## LGBM


### Основные параметры

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `n_estimators` | Количество итераций бустинга | Больше итераций → сложнее ансамбль, выше риск переобучения |
| `learning_rate` | Размер вклада очередного дерева | Меньше → обычно нужно больше деревьев; обучение более постепенное |
| `boosting_type` | Тип ансамбля: gbdt, dart, rf | gbdt — обычный бустинг; dart — случайное исключение деревьев при обучении; rf — режим случайного леса |
| `objective` | Функция потерь: regression, binary, multiclass и другие | Определяет оптимизируемую задачу, градиенты и гессианы |
| `metric` | Метрика качества: rmse, mae, auc, binary_logloss и другие | Используется для мониторинга и early stopping |
| `data_sample_strategy` | Стратегия отбора объектов: bagging, goss | Определяет, как выбирать наблюдения при обучении деревьев |
| `random_state` | Seed генератора случайных чисел | Помогает воспроизводить случайные процедуры |


 GOSS выбирается через data_sample_strategy='goss'. 



### Параметры сложности деревьев

В отличие от стандартного XGBoost, LightGBM по умолчанию использует стратегию leaf-wise: на каждом шаге разбивает лист с наибольшим улучшением функции потерь.

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `num_leaves` | Максимальное количество листьев одного дерева | Один из главных параметров. Больше листьев → сложнее модель и выше риск overfitting |
| `max_depth` | Максимальная глубина дерева; -1 — без ограничения | Ограничивает сложность и чрезмерно глубокие ветви |
| `min_child_samples (min_data_in_leaf)` | Минимальное количество наблюдений в листе (приближённо, с учётом гессианов) | Больше → меньше мелких листьев, ниже риск переобучения |
| `min_child_weight (min_sum_hessian_in_leaf)` | Минимальная сумма гессианов в листе | Больше → более консервативные разбиения |
| `min_split_gain (min_gain_to_split)` | Минимальное улучшение функции потерь для разбиения | Больше → меньше разбиений, проще деревья |
| `max_bin` | Максимальное число бинов для числовых признаков | Больше → потенциально точнее пороги, но выше затраты памяти и вычислений |
| `min_data_in_bin` | Минимальное количество объектов в одном бине | Больше → меньше слишком мелких бинов |
| `path_smooth` | Сглаживание предсказаний листьев с использованием значений родительских узлов | Больше → значения листьев консервативнее, меньше влияние небольших групп |



Главное отличие от XGBoost: для LightGBM очень важно подбирать не только max_depth, но и num_leaves, поскольку деревья растут несимметрично.

Например, num_leaves=31 не означает, что максимальная глубина дерева равна 5. Одна из ветвей может быть значительно глубже остальных.

### Регуляризация

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `reg_lambda (lambda_l2)` | L2-регуляризация весов листьев | Больше → значения листьев сильнее сжимаются |
| `reg_alpha (lambda_l1)` | L1-регуляризация весов листьев | Больше → некоторые значения листьев могут стать нулевыми |
| `min_split_gain` | Минимальное улучшение для создания разбиения | Больше → сложнее добавить новое разбиение |
| `min_child_samples` | Минимальное число объектов в листе | Больше → меньше риск подгонки под редкие наблюдения |
| `max_delta_step` | Ограничение величины выхода листа до умножения на learning_rate | Ограничивает величину обновления модели |


Здесь, как и в XGBoost, есть два разных механизма регуляризации:
1. Регуляризация структуры дерева — препятствует созданию лишних листьев.
2. Регуляризация значений листьев — уменьшает величину поправок, которые дерево вносит в предсказание ансамбля.

### Случайный выбор наблюдений и признаков

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `subsample (bagging_fraction)` | Доля объектов, случайно выбранных при bagging | 0.8 → 80% объектов. Работает при subsample_freq > 0 |
| `subsample_freq (bagging_freq)` | Частота пересоздания подвыборки объектов | 1 → новая подвыборка для каждого дерева; 0 → bagging отключён |
| `colsample_bytree (feature_fraction)` | Доля признаков для каждого дерева | 0.8 → 80% признаков дерева |
| `feature_fraction_bynode (colsample_bynode)` | Доля признаков при поиске разбиения конкретного узла | 0.5 → половина доступных признаков рассматривается в узле |
| `pos_bagging_fraction` | Доля объектов положительного класса | Позволяет отдельно управлять выборкой положительных объектов |
| `neg_bagging_fraction` | Доля объектов отрицательного класса | Позволяет отдельно управлять выборкой отрицательных объектов |
| `extra_trees` | Случайный порог для каждого признака при поиске split | Повышает случайность построения деревьев |



### Параметры построения деревьев и производительности

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `device_type` | Устройство: cpu, gpu, cuda | Определяет, где выполняются вычисления |
| `max_bin` | Максимальное количество бинов | Влияет на точность поиска порогов, память и скорость |
| `subsample_for_bin (bin_construct_sample_cnt)` | Число объектов для построения границ бинов | Больше → потенциально более представительные границы |
| `num_threads (n_jobs в sklearn API)` | Количество потоков CPU | Управляет параллелизмом |
| `max_bin_by_feature` | Отдельные ограничения числа бинов по признакам | Позволяет задавать разную детализацию числовых признаков |
| `force_col_wise` | Принудительное использование построения гистограмм по столбцам | Может быть эффективнее при большом количестве признаков |
| `force_row_wise` | Принудительное использование построения гистограмм по строкам | Может быть эффективнее при большом количестве объектов |
| `deterministic` | Режим более воспроизводимых вычислений на CPU | Уменьшает зависимость результата от числа потоков и порядка вычислений |
| `histogram_pool_size` | Ограничение памяти для кеша гистограмм (МБ) | Влияет на использование памяти и возможные повторные вычисления |


### Параметры GOSS

GOSS — Gradient-based One-Side Sampling.

Идея: наблюдения с большими по модулю градиентами информативнее для поиска следующего разбиения.

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `data_sample_strategy='goss'` | Включение GOSS | Меняет способ отбора объектов при обучении |
| `top_rate` | Доля объектов с наибольшими абсолютными градиентами | Больше → сохраняется больше сложных наблюдений |
| `other_rate` | Доля случайно отобранных объектов из оставшейся части | Больше → лучше представлена остальная выборка, но растут затраты |


Например, при top_rate=0.2 и other_rate=0.1 сохраняются 20% объектов с наибольшими абсолютными градиентами и случайные 10% из оставшихся 80%, то есть всего примерно 28% объектов. При вычислении статистик разбиений LightGBM корректирует вклад случайно отобранных наблюдений.

### Параметры категориальных признаков

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `categorical_feature` | Список категориальных признаков (в fit или Dataset) | Включает нативную обработку категорий без внешнего OHE |
| `max_cat_to_onehot` | Порог числа категорий для one-vs-other разбиений | Для низкой кардинальности используется другой способ поиска split |
| `max_cat_threshold` | Максимальное количество пороговых вариантов при поиске категориального разбиения | Ограничивает вычислительную сложность поиска |
| `cat_l2` | L2-регуляризация при поиске категориальных разбиений | Больше → осторожнее разбиения по категориям |
| `cat_smooth` | Сглаживание статистик категориальных разбиений | Уменьшает переобучение на редких категориях |
| `min_data_per_group` | Минимальное количество объектов в категориальной группе при обработке категорий | Уменьшает нестабильность для малочисленных групп |
| `max_cat_to_onehot` | Порог переключения механизма категориального split | Управляет обработкой низкокардинальных признаков |


### Особенность LightGBM: EFB

EFB (Exclusive Feature Bundling) объединяет признаки, которые редко принимают ненулевые значения одновременно, для уменьшения размерности.

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `enable_bundle` | Включает EFB | Может существенно ускорить обучение на разреженных данных |
| `is_enable_sparse` | Оптимизация хранения и обработки разреженных признаков | Уменьшает затраты памяти и вычислений на sparse-данных |
| `zero_as_missing` | Считать ли нули пропусками | Меняет интерпретацию нулевых значений; важно для разреженных матриц |


EFB особенно полезен при большом количестве разреженных признаков, например после OHE.


In [ ]:
# LGBM


import lightgbm as lgb


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42



# 1. Native: приводим категории к pandas category
X_train_native = X_train.copy()
X_valid_native = X_valid.copy()
X_test_native = X_test.copy()

for col in cat_cols:
    categories = pd.Index(
        X_train[col].dropna().unique()
    )

    dtype = pd.CategoricalDtype(
        categories=categories,
        ordered=False
    )

    for df in [
        X_train_native,
        X_valid_native,
        X_test_native
    ]:
        df[col] = df[col].astype(dtype)

# 2. One-Hot: обучаем encoder только на train
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            cat_cols
        ),
        ("num", "passthrough", num_cols)
    ],
    sparse_threshold=1.0
)

X_train_ohe = preprocessor.fit_transform(X_train)
X_valid_ohe = preprocessor.transform(X_valid)
X_test_ohe = preprocessor.transform(X_test)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


In [16]:

def objective(trial):

    encoding = trial.suggest_categorical(
        "encoding",
        ["native", "onehot"]
    )

    max_depth = trial.suggest_int(
        "max_depth", 3, 10
    )

    params = {
        "objective": "regression",
        "metric": "l1",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "n_estimators": 2000,

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "max_depth": max_depth,

        "num_leaves": trial.suggest_int(
            "num_leaves",
            4,
            min(128, 2 ** max_depth)
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples", 5, 100
        ),

        "min_child_weight": trial.suggest_float(
            "min_child_weight",
            1e-3, 10,
            log=True
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain", 0, 5
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        # Не забываем включить bagging
        "subsample_freq": 1,

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 100, log=True
        )
    }

    if encoding == "native":

        params.update({
            "max_cat_to_onehot": trial.suggest_int(
                "max_cat_to_onehot", 2, 10
            ),
            "max_cat_threshold": trial.suggest_int(
                "max_cat_threshold", 8, 64
            ),
            "cat_smooth": trial.suggest_float(
                "cat_smooth", 1, 100
            ),
            "cat_l2": trial.suggest_float(
                "cat_l2", 1e-2, 100, log=True
            )
        })

        X_tr = X_train_native
        X_val = X_valid_native

    else:
        X_tr = X_train_ohe
        X_val = X_valid_ohe

    model = lgb.LGBMRegressor(**params)

    model.fit(
        X_tr,
        y_train,
        eval_set=[(X_val, y_valid)],
        eval_metric="l1",
        categorical_feature=(
            cat_cols if encoding == "native"
            else "auto"
        ),
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=50,
                first_metric_only=True,
                verbose=False
            ),
            lgb.log_evaluation(period=0)
        ]
    )

    y_pred = model.predict(
        X_val,
        num_iteration=model.best_iteration_
    )

    mae = mean_absolute_error(
        y_valid, y_pred
    )

    trial.set_user_attr(
        "best_iteration",
        model.best_iteration_
    )

    return mae


In [17]:

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

# Обеспечиваем первоначальную проверку обоих способов
study.enqueue_trial({"encoding": "native"})
study.enqueue_trial({"encoding": "onehot"})

study.optimize(
    objective,
    n_trials=10,
    show_progress_bar=True
)

print("Best MAE:", study.best_value)
print("Best encoding:", study.best_params["encoding"])
print("Best params:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

print(
    "Best iteration:",
    study.best_trial.user_attrs["best_iteration"]
)


  0%|          | 0/10 [00:00<?, ?it/s]

/Users/irinakharkova/my_projects/ml_course/2 семинар линейнай регрессия/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/irinakharkova/my_projects/ml_course/2 семинар линейнай регрессия/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/irinakharkova/my_projects/ml_course/2 семинар линейнай регрессия/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/irinakharkova/my_projects/ml_course/2 семи

Best MAE: 0.3249640000991172
Best encoding: native
Best params:
encoding: native
max_depth: 8
learning_rate: 0.136080647750805
num_leaves: 104
min_child_samples: 22
min_child_weight: 3.7173717478250525
min_split_gain: 2.6967112095782535
subsample: 0.922976062065625
colsample_bytree: 0.9584365199693973
reg_alpha: 7.278322158604965e-06
reg_lambda: 0.0035502556123130774
max_cat_to_onehot: 4
max_cat_threshold: 32
cat_smooth: 81.98346182632682
cat_l2: 27.728241828010624
Best iteration: 79


In [18]:
trials_df = study.trials_dataframe()

comparison = (
    trials_df
    .query("state == 'COMPLETE'")
    .groupby("params_encoding")
    .agg(
        best_mae=("value", "min"),
        mean_mae=("value", "mean"),
        n_trials=("value", "count")
    )
    .sort_values("best_mae")
)

display(comparison)

,best_mae,mean_mae,n_trials
params_encoding,,,
native,0.324964,0.353804,6
onehot,0.362074,0.385315,4


In [19]:
import optuna.visualization as vis

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()
vis.plot_slice(study).show()

/var/folders/dp/9vgxx_1j3y33c79jyd72bbgm0000gn/T/ipykernel_32081/1660780027.py:4: UserWarning: Some regimes for parameter `num_leaves` have less than 2 trials. The importance of the parameter may be inaccurate.
  vis.plot_param_importances(study).show()


In [20]:

from sklearn.base import clone
from sklearn.metrics import mean_squared_error

best_params = study.best_params.copy()
best_encoding = best_params.pop("encoding")

best_n_estimators = (
    study.best_trial.user_attrs["best_iteration"]
)

X_full = pd.concat([X_train, X_valid])
y_full = pd.concat([y_train, y_valid])

if best_encoding == "native":
    X_full_final = X_full.copy()
    X_test_final = X_test.copy()

    for col in cat_cols:
        dtype = pd.CategoricalDtype(
            categories=pd.Index(
                X_full[col].dropna().unique()
            ),
            ordered=False
        )
        X_full_final[col] = X_full[col].astype(dtype)
        X_test_final[col] = X_test[col].astype(dtype)

    final_cat_features = cat_cols

else:
    final_encoder = clone(preprocessor)

    X_full_final = final_encoder.fit_transform(X_full)
    X_test_final = final_encoder.transform(X_test)

    final_cat_features = "auto"

final_model = lgb.LGBMRegressor(
    **best_params,
    objective="regression",
    n_estimators=best_n_estimators,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

final_model.fit(
    X_full_final,
    y_full,
    categorical_feature=final_cat_features
)

y_pred_test = final_model.predict(X_test_final)

print("Encoding:", best_encoding)
print("Test MAE:",
      mean_absolute_error(y_test, y_pred_test))
print("Test RMSE:",
      np.sqrt(mean_squared_error(y_test, y_pred_test)))


Encoding: native
Test MAE: 0.3032586730945374
Test RMSE: 0.6081979082734974


# CatBoost

### Основные параметры

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `iterations (n_estimators)` | Максимальное количество итераций бустинга | Больше итераций → потенциально сложнее модель, выше риск overfitting |
| `learning_rate` | Размер вклада очередного дерева | Меньше → обычно требуется больше итераций |
| `loss_function` | Функция потерь: RMSE, MAE, Logloss, MultiClass и другие | Определяет оптимизируемую задачу |
| `eval_metric` | Метрика для выбора лучшей итерации и контроля переобучения | Используется при early stopping и use_best_model |
| `custom_metric` | Дополнительные метрики мониторинга | Позволяет отслеживать качество, не меняя оптимизируемый loss |
| `boosting_type` | Схема бустинга: Plain, Ordered | Определяет способ вычисления приближений при обучении |
| `random_seed (random_state)` | Seed генератора случайных чисел | Помогает воспроизводить результаты |


 boosting_type в CatBoost — не аналог boosting_type в LightGBM.

В CatBoost:
- Plain — классический градиентный бустинг.
- Ordered — схема, использующая перестановки объектов для уменьшения prediction shift, возникающего при вычислении обучающих статистик и градиентов.
Ordered boosting часто полезен на небольших датасетах, но может быть вычислительно дороже. 



### Параметры сложности деревьев

CatBoost по умолчанию строит симметричные деревья (oblivious trees).

На каждом уровне для всех узлов выбирается одно и то же условие разбиения.

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `depth (max_depth)` | Глубина дерева | Больше → сложнее взаимодействия признаков, выше риск переобучения |
| `grow_policy` | Стратегия роста: SymmetricTree, Depthwise, Lossguide | Определяет форму деревьев |
| `min_data_in_leaf` | Минимальное количество объектов в листе | Больше → меньше разбиений; работает только с Depthwise и Lossguide |
| `max_leaves (num_leaves)` | Максимальное количество листьев | Больше → сложнее деревья; работает только с Lossguide |
| `border_count (max_bin)` | Максимальное число границ разбиения для числовых признаков | Больше → детальнее поиск порогов, но выше затраты |
| `feature_border_type` | Алгоритм выбора границ квантования числовых признаков | Меняет расположение допустимых порогов |
| `score_function` | Способ оценки качества кандидатов на разбиение: Cosine, L2, NewtonCosine, NewtonL2 | Влияет на выбор признака и порога |
| `leaf_estimation_method` | Метод расчёта значений листьев: Gradient, Newton, Exact | Определяет алгоритм оптимизации значений листьев |
| `leaf_estimation_iterations` | Количество шагов оптимизации значений листьев одного дерева | Больше → точнее оптимизация листьев, но выше вычислительная стоимость |


Для grow_policy действуют следующие различия:

| Значение | Как строится дерево | Особенности |
| --- | --- | --- |
| `SymmetricTree` | На одном уровне все узлы разбиваются по одному условию | Быстрое предсказание, регулярная структура |
| `Depthwise` | На каждом уровне разные узлы могут разбиваться по разным условиям | Более гибкая структура |
| `Lossguide` | Разбивается лист с наибольшим улучшением loss | Позволяет сосредоточить сложность на отдельных областях данных |


При SymmetricTree дерево глубины \(d\) имеет \(2^d\) листьев.

Например, depth=6 соответствует 64 листьям полного симметричного дерева.

При Depthwise и Lossguide количество листьев уже не обязано равняться \(2^d\). 




### Регуляризация

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `l2_leaf_reg (reg_lambda)` | L2-регуляризация значений листьев | Больше → значения листьев сильнее сжимаются |
| `random_strength` | Масштаб случайного шума, добавляемого к оценкам кандидатов на split | Больше → выбор разбиений становится более случайным |
| `depth` | Глубина дерева | Меньше → ограничивается сложность взаимодействий признаков |
| `min_data_in_leaf` | Минимальное количество объектов в листе | Больше → меньше подгонка под небольшие группы; только Depthwise/Lossguide |
| `bagging_temperature` | Степень случайности весов объектов при Bayesian bootstrap | Больше → сильнее различаются веса объектов |
| `model_size_reg` | Регуляризация размера модели, связанная с использованием категориальных признаков | Может ограничивать сложность модели и её размер |
| `penalties_coefficient` | Общий множитель штрафов за использование признаков | Позволяет регулировать дополнительные feature penalties |
| `first_feature_use_penalties` | Штраф за первое использование признака в дереве | Может уменьшать вероятность выбора определённых признаков |
| `per_object_feature_penalties` | Штраф за первое использование признака на пути отдельного объекта | Влияет на выбор разбиений с учётом объектов |


Особенно интересен random_strength.

CatBoost оценивает потенциальное улучшение от каждого разбиения. Затем в оценку добавляется случайный шум.

Это позволяет иногда выбирать не строго лучшее разбиение по обучающей выборке и тем самым снижать риск переобучения.

Важно: random_strength влияет на выбор структуры дерева, а l2_leaf_reg — на значения листьев. Поддержка некоторых механизмов регуляризации зависит от CPU/GPU и функции потерь. 




### Случайный выбор наблюдений и признаков

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `bootstrap_type` | Способ сэмплирования и назначения весов объектам | Определяет механизм случайности в обучении |
| `subsample` | Доля объектов для bootstrap | 0.8 → используется примерно 80% объектов; применим к определённым типам bootstrap |
| `bagging_temperature` | Интенсивность случайных весов при Bayesian bootstrap | Больше → веса объектов сильнее различаются |
| `sampling_frequency` | Частота обновления весов/выборки: PerTree, PerTreeLevel | Определяет, насколько часто пересоздаётся выборка или веса |
| `sampling_unit` | Единица сэмплирования: Object, Group | Позволяет сэмплировать объекты или группы |
| `rsm (colsample_bylevel)` | Доля признаков, случайно выбираемых при поиске очередного разбиения | 0.8 → рассматривается примерно 80% признаков |
| `mvs_reg` | Регуляризация схемы MVS sampling | Регулирует баланс между importance sampling и равномерным случайным отбором |


### Виды bootstrap в CatBoost

| bootstrap_type | Как работает | Особенности |
| --- | --- | --- |
| `Bayesian` | Объектам назначаются случайные положительные веса | Добавляет случайность, но не обязательно уменьшает число обрабатываемых объектов |
| `Bernoulli` | Каждый объект независимо включается в подвыборку с заданной вероятностью | Может ускорять обучение |
| `MVS` | Объекты выбираются с вероятностями, связанными с величинами градиентов | Снижает вычислительные затраты, стараясь сохранить информативность выборки |
| `Poisson` | Объектам назначаются случайные веса из распределения Пуассона | Поддерживается на GPU |
| `No` | Bootstrap отключён | Отсутствует случайность от bootstrap |


### Параметры построения деревьев и производительности

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `task_type` | Устройство: CPU, GPU | Определяет, где выполняется обучение |
| `devices` | Номера GPU | Позволяет выбрать конкретные GPU |
| `thread_count` | Количество потоков CPU | Влияет на параллелизм |
| `border_count` | Максимальное число границ квантования | Определяет детализацию поиска порогов |
| `feature_border_type` | Метод выбора границ квантования | Влияет на расположение числовых порогов |
| `gpu_ram_part` | Доля памяти GPU, которую разрешено использовать | Управляет ограничением видеопамяти |
| `used_ram_limit` | Ориентировочное ограничение оперативной памяти | Ограничивает часть потребления памяти, прежде всего CTR-расчётов |
| `gpu_cat_features_storage` | Способ хранения категориальных признаков на GPU | Влияет на использование GPU/CPU памяти |
| `nan_mode` | Обработка пропусков: Min, Max, Forbidden | Определяет положение пропусков при поиске разбиений |
| `allow_writing_files` | Разрешение записи служебных файлов обучения | Управляет сохранением логов и других файлов |
| `verbose` | Частота вывода информации об обучении | Управляет логированием |


### Параметры категориальных признаков

Это одно из главных преимуществ CatBoost: категориальные признаки можно передавать напрямую, без предварительного OneHotEncoder или TargetEncoder.

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `cat_features` | Список категориальных признаков (имена или индексы) | Включает нативную обработку категорий |
| `one_hot_max_size` | Максимальное число уникальных категорий для one-hot-кодирования | Для признаков с числом категорий не выше порога используется one-hot |
| `simple_ctr` | Типы CTR для отдельных категориальных признаков | Определяет, как рассчитываются целевые статистики категорий |
| `combinations_ctr` | Типы CTR для комбинаций категориальных признаков | Позволяет учитывать взаимодействия категорий |
| `max_ctr_complexity` | Максимальное число признаков в категориальной комбинации для CTR | Больше → сложнее взаимодействия, выше затраты памяти и риск overfitting |
| `ctr_leaf_count_limit` | Ограничение числа различных категориальных значений для CTR | Может уменьшить размер модели |
| `ctr_target_border_count` | Число границ дискретизации целевой переменной при CTR | Влияет на детализацию целевых статистик |
| `counter_calc_method` | Метод расчёта счётчиков категорий при применении модели | Определяет использование доступных данных для counter-статистик |


Различие с LightGBM:
- LightGBM ищет оптимальные разбиения по группам категорий, используя градиенты и гессианы.
- CatBoost может превращать категории и их комбинации в числовые статистики (CTR), используя упорядоченные расчёты для борьбы с target leakage.
При этом CatBoost не обязательно применяет CTR ко всем категориям: для малой кардинальности используется one-hot-подобная обработка. 




### Параметры Ordered Boosting и контроля переобучения

| Параметр | Что означает | Как влияет |
| --- | --- | --- |
| `boosting_type` | Plain или Ordered | Ordered уменьшает prediction shift, но может работать медленнее |
| `fold_permutation_block` | Размер блоков при случайных перестановках объектов | Больше → меньше перемешивание, возможное ускорение |
| `has_time` | Использовать порядок объектов без случайных перестановок на соответствующих этапах | Важно, когда порядок наблюдений имеет содержательный смысл |
| `fold_len_multiplier` | Управляет длиной префиксов при ordered-вычислениях | Влияет на время обучения, память и потенциально качество |
| `od_type` | Тип детектора переобучения: IncToDec, Iter | Определяет условие обнаружения переобучения |
| `od_wait` | Число итераций ожидания для детектора переобучения | Больше → модель дольше обучается без улучшения |
| `od_pval` | Порог значимости для детектора IncToDec | Влияет на чувствительность обнаружения переобучения |
| `early_stopping_rounds` | Удобный способ включить остановку после заданного числа итераций без улучшения | Помогает остановить обучение до переобучения |
| `use_best_model` | Сохранить только деревья до лучшей итерации на validation | Может уменьшить размер итогового ансамбля |


In [ ]:

# pip install catboost optuna scikit-learn

import catboost as cb

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.base import clone

RANDOM_STATE = 42



# Native CatBoost
X_train_native = X_train.copy()
X_valid_native = X_valid.copy()
X_test_native = X_test.copy()


# CatBoost принимает категории как строки или целые числа,
# поэтому пропуски заменяем отдельной категорией
for col in cat_cols:
    for df in [
        X_train_native,
        X_valid_native,
        X_test_native
    ]:
        df[col] = (
            df[col]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

# External One-Hot Encoding
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            cat_cols
        ),
        ("num", "passthrough", num_cols)
    ],
    sparse_threshold=1.0
)

X_train_ohe = preprocessor.fit_transform(X_train)
X_valid_ohe = preprocessor.transform(X_valid)
X_test_ohe = preprocessor.transform(X_test)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


In [22]:

def objective(trial):

    encoding = trial.suggest_categorical(
        "encoding",
        ["native", "onehot"]
    )

    bootstrap_type = trial.suggest_categorical(
        "bootstrap_type",
        ["Bayesian", "Bernoulli"]
    )

    params = {
        "loss_function": "RMSE",
        "eval_metric": "MAE",
        "iterations": 2000,
        "random_seed": RANDOM_STATE,
        "thread_count": -1,
        "verbose": False,
        "allow_writing_files": False,
        "bootstrap_type": bootstrap_type,

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "depth": trial.suggest_int(
            "depth", 3, 10
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 0.01, 100, log=True
        ),

        "random_strength": trial.suggest_float(
            "random_strength", 0.001, 10, log=True
        ),

        "rsm": trial.suggest_float(
            "rsm", 0.6, 1.0
        ),
    }

    if bootstrap_type == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float(
            "bagging_temperature", 0, 5
        )
    else:
        params["subsample"] = trial.suggest_float(
            "subsample", 0.6, 1.0
        )

    if encoding == "native":

        params["one_hot_max_size"] = trial.suggest_int(
            "one_hot_max_size", 2, 10
        )

        X_tr = X_train_native
        X_val = X_valid_native
        categorical_features = cat_cols

    else:

        X_tr = X_train_ohe
        X_val = X_valid_ohe
        categorical_features = None

    model = cb.CatBoostRegressor(**params)

    model.fit(
        X_tr,
        y_train,
        cat_features=categorical_features,
        eval_set=(X_val, y_valid),
        early_stopping_rounds=50,
        use_best_model=True,
        verbose=False
    )

    y_pred = model.predict(X_val)

    mae = mean_absolute_error(y_valid, y_pred)

    trial.set_user_attr(
        "best_iteration", model.best_iteration_
    )

    return mae


In [23]:

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE
    )
)

study.enqueue_trial({"encoding": "native"})
study.enqueue_trial({"encoding": "onehot"})

study.optimize(
    objective,
    n_trials=10,
    show_progress_bar=True
)

print("Best MAE:", study.best_value)
print("Best encoding:", study.best_params["encoding"])
print("Best params:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

print(
    "Best iteration:",
    study.best_trial.user_attrs["best_iteration"]
)


  0%|          | 0/10 [00:00<?, ?it/s]

Best MAE: 0.28261958795258935
Best encoding: native
Best params:
encoding: native
bootstrap_type: Bernoulli
learning_rate: 0.08960785365368121
depth: 7
l2_leaf_reg: 0.04207988669606637
random_strength: 0.004207053950287938
rsm: 0.6232334448672797
subsample: 0.9464704583099741
one_hot_max_size: 7
Best iteration: 753


In [24]:
trials_df = study.trials_dataframe()

comparison = (
    trials_df
    .query("state == 'COMPLETE'")
    .groupby("params_encoding")
    .agg(
        best_mae=("value", "min"),
        mean_mae=("value", "mean"),
        n_trials=("value", "count")
    )
    .sort_values("best_mae")
)

display(comparison)

,best_mae,mean_mae,n_trials
params_encoding,,,
native,0.282620,0.305117,7
onehot,0.292161,0.336686,3


In [25]:
best_trials = {}

for encoding in ["native", "onehot"]:

    trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
        and t.params["encoding"] == encoding
    ]

    best = min(trials, key=lambda t: t.value)
    best_trials[encoding] = best

    print(f"\nEncoding: {encoding}")
    print("MAE:", best.value)
    print("Best iteration:", best.user_attrs["best_iteration"])
    print("Params:", best.params)


Encoding: native
MAE: 0.28261958795258935
Best iteration: 753
Params: {'encoding': 'native', 'bootstrap_type': 'Bernoulli', 'learning_rate': 0.08960785365368121, 'depth': 7, 'l2_leaf_reg': 0.04207988669606637, 'random_strength': 0.004207053950287938, 'rsm': 0.6232334448672797, 'subsample': 0.9464704583099741, 'one_hot_max_size': 7}

Encoding: onehot
MAE: 0.2921605544363303
Best iteration: 1882
Params: {'encoding': 'onehot', 'bootstrap_type': 'Bayesian', 'learning_rate': 0.1827602783178572, 'depth': 9, 'l2_leaf_reg': 0.07068974950624607, 'random_strength': 0.005337032762603957, 'rsm': 0.6733618039413735, 'bagging_temperature': 1.5212112147976886}


In [26]:

X_full = pd.concat([X_train, X_valid])
y_full = pd.concat([y_train, y_valid])

results = []
final_models = {}

for encoding, best_trial in best_trials.items():

    params = best_trial.params.copy()

    params.pop("encoding")

    # Количество деревьев после early stopping
    n_iterations = (
        best_trial.user_attrs["best_iteration"] + 1
    )

    if encoding == "native":

        X_full_final = pd.concat([
            X_train_native, X_valid_native
        ])
        X_test_final = X_test_native

        categorical_features = cat_cols

    else:

        final_encoder = clone(preprocessor)

        X_full_final = final_encoder.fit_transform(X_full)
        X_test_final = final_encoder.transform(X_test)

        categorical_features = None

    model = cb.CatBoostRegressor(
        **params,
        iterations=n_iterations,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        thread_count=-1,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_full_final,
        y_full,
        cat_features=categorical_features
    )

    y_pred = model.predict(X_test_final)

    results.append({
        "encoding": encoding,
        "test_mae": mean_absolute_error(
            y_test, y_pred
        ),
        "test_rmse": np.sqrt(
            mean_squared_error(y_test, y_pred)
        ),
        "n_trees": model.tree_count_
    })

    final_models[encoding] = model

results_df = pd.DataFrame(results)
display(results_df.sort_values("test_mae"))


,encoding,test_mae,test_rmse,n_trees
0,native,0.275907,0.618010,754
1,onehot,0.284574,0.725076,1883


In [27]:
import optuna.visualization as vis

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

vis.plot_slice(
    study,
    params=["depth", "learning_rate", "encoding"]
).show()

## XGBoost / LightGBM / CatBoost

Все три — градиентный бустинг над деревьями, но отличаются структурой дерева, оптимизацией и обработкой признаков.

| Аспект | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| Типичный рост дерева | depth-wise (есть другие режимы) | leaf-wise | symmetric / oblivious по умолчанию |
| Ключевые особенности | Регуляризация, градиент/Гессиан, histogram | Histogram, leaf-wise, GOSS/EFB в соответствующих режимах | Ordered boosting, специально организованная обработка категорий |
| Категориальные признаки | Native categorical (`enable_categorical=True`), либо OHE | Native categories, либо OHE | Native categories, в том числе target-statistics |
| Главный риск | Настройка сложности, overfit | Глубокие пути из-за leaf-wise роста | Время/ресурсы; настройки и режим обработки категорий |



Документация: [XGBoost: categorical features](https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html), [LightGBM: features](https://lightgbm.readthedocs.io/en/stable/Features.html), [CatBoost: categorical features](https://catboost.ai/en/docs/concepts/algorithm-main-stages_cat-to-numberic).

# Задачи 

## GB

### Задача 1. 

Рассматривается задача прогнозирования месячной выручки пяти компаний:

$$
y=(10,14,18,22,36).
$$

Используется градиентный бустинг с квадратичной функцией потерь:

$$
L(y,F)=\frac12(y-F)^2.
$$

1. Найдите оптимальное константное предсказание $F_0(x)$.
2. Рассчитайте антиградиенты для всех пяти наблюдений.

---

### Задача 2. Один шаг градиентного бустинга

Пусть текущие предсказания модели:

$$
F_0=(10,10,10,10),
$$

а истинные значения:

$$
y=(8,12,16,20).
$$

Очередное дерево разбивает наблюдения на два листа:

$$
R_1=\{1,2\}, \qquad R_2=\{3,4\}.
$$

Используется квадратичная функция потерь:

$$
L(y,F)=\frac12(y-F)^2
$$

и learning rate $\eta=0.2$.

1. Найдите антиградиенты для всех наблюдений.
2. Определите оптимальное значение каждого листа.
3. Рассчитайте предсказания после добавления дерева.
4. Найдите MSE до и после обновления модели.

---

### Задача 3. Влияние learning rate

Пусть для некоторого объекта текущее предсказание равно:

$$
F_0(x)=100,
$$

а очередное дерево предсказывает:

$$
h_1(x)=40.
$$

Рассмотрите три значения learning rate:

$$
\eta\in\{0.01,0.1,0.5\}.
$$

1. Найдите новое предсказание для каждого значения $\eta$.
2. Почему маленький learning rate обычно требует большего количества деревьев?
3. Можно ли утверждать, что уменьшение learning rate всегда улучшает качество на тестовой выборке?

---

### Задача 4. 

Исследователь предлагает обучить 100 деревьев независимо на исходной целевой переменной, а затем сложить их предсказания с коэффициентом 0.01.

1. Является ли предложенный алгоритм градиентным бустингом?
2. Чем он отличается от стандартного градиентного бустинга?
3. Почему каждое следующее дерево в бустинге зависит от предыдущих?
4. Можно ли обучать все деревья стандартного градиентного бустинга независимо и параллельно?

---

### Задача 5. Антиградиенты разных функций потерь

Рассмотрим одно наблюдение:

$$
y=10, \qquad F(x)=7.
$$

Найдите антиградиенты для следующих функций потерь:

1. Квадратичная ошибка:

   $$
   L(y,F)=\frac12(y-F)^2.
   $$

2. Абсолютная ошибка:

   $$
   L(y,F)=|y-F|.
   $$

3. Функция потерь:

   $$
   L(y,F)=\frac14(y-F)^4.
   $$

4. Объясните, как выбор функции потерь влияет на чувствительность бустинга к большим ошибкам.

---

### Задача 6. Градиент логистической функции потерь

Для бинарной классификации используется:

$$
p_i=\sigma(F_i)=\frac{1}{1+e^{-F_i}},
$$

где $F_i$ — значение модели до применения сигмоиды.

Функция потерь:

$$
L(y_i,F_i)=-y_i\log p_i-(1-y_i)\log(1-p_i).
$$

1. Докажите, что:

   $$
   \frac{\partial L}{\partial F_i}=p_i-y_i.
   $$

2. Найдите антиградиенты для наблюдений:

   | Объект | $y_i$ | $p_i$ |
   |:---:|---:|---:|
   | A | 1 | 0.2 |
   | B | 1 | 0.9 |
   | C | 0 | 0.8 |
   | D | 0 | 0.1 |

3. Для каких объектов модель получает наибольший по модулю сигнал к корректировке?
4. Почему в градиентном бустинге для классификации нельзя просто прибавлять предсказания деревьев непосредственно к вероятностям?

---

### Задача 7. Начальное приближение в классификации

В выборке 100 клиентов, из которых 20 допустили дефолт.

Бустинг минимизирует бинарный log-loss, а $F(x)$ задаёт log-odds дефолта:

$$
F(x)=\log\frac{p(x)}{1-p(x)}.
$$

1. Найдите оптимальную начальную вероятность $p_0$.
2. Вычислите оптимальное начальное приближение $F_0$.
3. Почему $F_0\neq 0.2$?
4. Как изменится начальное приближение, если доля дефолтов вырастет до 50%?

---

### Задача 8. Градиент и гессиан

Для бинарного log-loss используется выражение:

$$
L(y,F)=\log(1+e^F)-yF.
$$

1. Найдите первую и вторую производные функции потерь по $F$.
2. Рассчитайте их для $y=1$ и $p=0.8$, где $p=\sigma(F)$.
3. Для чего XGBoost использует вторые производные при построении деревьев?

---

### Задача 9. Выбор оптимального разбиения

На очередной итерации бустинга получены антиградиенты:

| $x_i$ | 1 | 2 | 3 | 4 | 5 | 6 |
|:---|---:|---:|---:|---:|---:|---:|
| $r_i$ | 2 | 3 | 4 | -2 | -3 | -4 |

Дерево имеет глубину 1 и обучается методом наименьших квадратов на антиградиентах.

Рассмотрите три возможных порога:

$$
x\leq 2.5,\qquad x\leq 3.5,\qquad x\leq 4.5.
$$

1. Найдите средние антиградиенты в каждом листе для каждого разбиения.
2. Вычислите сумму квадратов отклонений внутри листьев.
3. Выберите оптимальный порог.
4. Объясните, почему разбиение, хорошо разделяющее антиградиенты, полезно для бустинга.

---

### Задача 10. Оптимальный вес листа XGBoost

В XGBoost для одного листа рассматривается приближение функции потерь:

$$
\widetilde L(w)=Gw+\frac12(H+\lambda)w^2,
$$

где:

- $G=\sum_i g_i$ — сумма градиентов;
- $H=\sum_i h_i$ — сумма вторых производных;
- $\lambda\geq 0$ — коэффициент L2-регуляризации;
- $w$ — значение листа.

1. Найдите оптимальное значение $w^*$.
2. Вычислите его при $G=-12$, $H=6$ для:

   $$
   \lambda\in\{0,2,6\}.
   $$

3. Объясните влияние $\lambda$ на величину обновления.
4. Почему регуляризация особенно важна для листьев с небольшим числом наблюдений или малой суммарной кривизной?

---

### Задача 11. Gain разбиения в XGBoost

Для одного узла известны значения:

$$
G=-10,\qquad H=8.
$$

После разбиения:

$$
G_L=-8,\qquad H_L=3,
$$

$$
G_R=-2,\qquad H_R=5.
$$

Пусть:

$$
\lambda=1,\qquad \gamma=0.5.
$$

Используйте формулу:

$$
Gain=
\frac12\left(
\frac{G_L^2}{H_L+\lambda}
+
\frac{G_R^2}{H_R+\lambda}
-
\frac{G^2}{H+\lambda}
\right)-\gamma.
$$

1. Вычислите Gain.
2. Следует ли выполнять разбиение?
3. Как изменится решение при $\gamma=2$?
4. Какую роль играют $\lambda$ и $\gamma$, и в чём их различие?

---

### Задача 12. Регуляризация структуры дерева

Рассматриваются две модели градиентного бустинга:

| Параметр | Модель A | Модель B |
|:---|---:|---:|
| max_depth | 3 | 10 |
| learning_rate | 0.05 | 0.2 |
| n_estimators | 500 | 500 |
| min_child_weight | 10 | 1 |
| reg_lambda | 5 | 0 |

1. Какая модель потенциально более склонна к переобучению?
2. Как каждый параметр влияет на сложность и обучение модели?
3. Можно ли по этим параметрам однозначно предсказать, какая модель даст меньшую test MSE?
4. Предложите эксперимент для проверки гипотезы о переобучении.

---

### Задача 13. Leaf-wise против level-wise

Рассматриваются два способа построения дерева:

- **Алгоритм A:** расширяет дерево по уровням (level-wise).
- **Алгоритм B:** на каждом шаге разбивает лист с максимальным улучшением целевой функции (leaf-wise).

1. Какой подход характерен для стандартной стратегии роста деревьев LightGBM?
2. Почему leaf-wise может быстрее уменьшать training loss?
3. Какой подход потенциально более склонен к переобучению на небольших выборках?
4. Какие ограничения можно наложить на leaf-wise дерево?

---

### Задача 14. Симметричные деревья CatBoost

Пусть строится симметричное дерево глубины 3.

На каждом уровне дерева используется одно и то же условие разбиения для всех узлов этого уровня.

1. Сколько у дерева листьев?
2. Сколько различных условий разбиения выбирается по уровням?
3. Чем симметричное дерево отличается от обычного бинарного дерева?


---

### Задача 15. Диагностика переобучения

Получены значения RMSE на обучающей и валидационной выборках:

| Число деревьев | Train RMSE | Validation RMSE |
|---:|---:|---:|
| 50 | 12.0 | 13.0 |
| 100 | 9.0 | 10.5 |
| 200 | 6.5 | 9.2 |
| 400 | 4.0 | 9.8 |
| 800 | 2.0 | 11.5 |

1. Какое количество деревьев следует выбрать по validation RMSE?
2. На каком этапе проявляется переобучение?
3. Почему train RMSE продолжает снижаться?
4. Почему нельзя подбирать число деревьев на test set?

---


### Задача 16. Выбор гиперпараметров по результатам эксперимента

Для прогнозирования месячного спроса получены результаты:

| Модель | max_depth | learning_rate | n_estimators | Train RMSE | Validation RMSE |
|:---:|---:|---:|---:|---:|---:|
| A | 2 | 0.05 | 300 | 15 | 18 |
| B | 5 | 0.05 | 500 | 10 | 14 |
| C | 10 | 0.1 | 800 | 3 | 19 |
| D | 5 | 0.01 | 100 | 21 | 23 |
| E | 3 | 0.05 | 500 | 12 | 13 |

1. Выберите лучшую модель по validation RMSE.
2. Какие модели демонстрируют признаки underfitting и overfitting?
3. Какие гиперпараметры можно изменить для улучшения модели C?
4. Почему нельзя сделать вывод о причинном влиянии `max_depth` на качество по этой таблице?
5. Предложите корректный план дальнейшего подбора гиперпараметров.


# BVD

Пусть $Y=f(x)+\varepsilon$, где $\mathbb E[\varepsilon\mid x]=0$, $\operatorname{Var}(\varepsilon\mid x)=\sigma^2$, а шум **нового тестового наблюдения** независим от обучающей выборки $D$. Модель $\hat f_D(x)$ обучена на $D$.

$$
\boxed{
\mathbb E_{D,\varepsilon}\left[(Y-\hat f_D(x))^2\mid x\right]
=
\underbrace{\left(\mathbb E_D[\hat f_D(x)]-f(x)\right)^2}_{\mathrm{Bias}^2(x)}
+
\underbrace{\operatorname{Var}_D\!\left(\hat f_D(x)\right)}_{\mathrm{Variance}(x)}
+
\underbrace{\sigma^2}_{\mathrm{Noise}}
}
$$

Такое разложение только для **квадратичной функции потерь**.


### Задача 1.

Истинная ожидаемая стоимость квартиры с заданными характеристиками — **12 млн руб.** Модель обучили на пяти независимых выборках и получили прогнозы (млн руб.):

$$
10,\quad 11,\quad 12,\quad 13,\quad 14.
$$

Известно: $\sigma^2=1\ \text{(млн руб.)}^2$.

1. Найдите среднее предсказание, bias и $\mathrm{Bias}^2$.
2. Вычислите дисперсию пяти прогнозов, используя деление на **5**.
3. Найдите MSE по формуле разложения, считая приведённые пять исходов равновероятным распределением предсказаний.

### Задача 2. 

| Характеристика | Модель A | Модель B |
|:--|--:|--:|
| Bias | 2 | 0.5 |
| Variance | 1 | 6 |
| Noise variance | 2 | 2 |

1. Найдите ожидаемую MSE каждой модели.
2. Выберите модель по критерию минимальной MSE.
3. Объясните, почему меньший bias ещё не гарантирует лучшего прогноза.

### Задача 3. 

Дано:

$$
\mathrm{MSE}=15,\qquad
\mathrm{Bias}^2=4,\qquad
\sigma^2=3.
$$

1. Найдите variance.
2. Найдите MSE после уменьшения variance вдвое при неизменных остальных компонентах.
3. Может ли ожидаемая MSE быть меньше 3 при данных допущениях? Почему?

### Задача 4. 

Пусть $f(x)=5$, а предсказание случайно:

| $\hat f_D(x)$ | 3 | 5 | 7 |
|:--|--:|--:|--:|
| Вероятность | 0.25 | 0.50 | 0.25 |

1. Найдите $\mathbb E_D[\hat f_D(x)]$, bias и variance.
2. Найдите MSE нового наблюдения при $\sigma^2=2$.
3. Почему несмещённая модель не обязана иметь нулевую ошибку?

### Задача 5.

Пусть

$$
Y_i=\mu+\varepsilon_i,
$$

где наблюдения независимы, $\mathbb E[\varepsilon_i]=0$, $\operatorname{Var}(\varepsilon_i)=16$.

Сравним прогнозы нового $Y$:

- **A:** $\hat\mu_A=Y_1$;
- **B:** $\hat\mu_B=\bar Y$, где $n=4$.

1. Найдите bias и variance обоих прогнозов относительно $\mu$.
2. Найдите ожидаемую MSE прогноза нового $Y$.
3. Что произойдёт с MSE модели B при $n\to\infty$?


### Задача 6.

Для фиксированного $x$ докажите:

$$
\mathbb E_{D,\varepsilon}[(Y-\hat f_D(x))^2]
=
(f(x)-\mathbb E_D[\hat f_D(x)])^2
+
\operatorname{Var}_D(\hat f_D(x))
+
\sigma^2.
$$

Укажите, что нужно добавить и вычесть внутри квадрата, и почему исчезают смешанные слагаемые.

Какие предположения об $\varepsilon$ существенны?

### Задача 7. Смещённая оценка среднего

Пусть $X_1,\ldots,X_n$ независимы,

$$
\mathbb E[X_i]=\mu,\qquad
\operatorname{Var}(X_i)=\sigma^2.
$$

Рассмотрим

$$
\hat\mu_\alpha=\alpha\bar X,
\qquad \alpha\in[0,1].
$$

1. Выведите bias, variance и MSE оценки относительно $\mu$.
2. Найдите оптимальное $\alpha$ как функцию $n,\mu,\sigma^2$.
3. Почему оптимальная по MSE оценка может быть смещённой? 


### Задача 8. Усреднение независимых моделей

Даны $B$ независимых моделей с одинаковыми

$$
\mathbb E_D[\hat f_b(x)]=m,\qquad
\operatorname{Var}_D(\hat f_b(x))=v.
$$

Ансамбль:

$$
\hat f_{\mathrm{ens}}(x)
=
\frac1B\sum_{b=1}^B\hat f_b(x).
$$

1. Найдите bias и variance ансамбля.
2. Во сколько раз уменьшится variance при $B=10$?
3. Почему усреднение не обязано уменьшать bias?

### Задача 9. Коррелированные деревья Random Forest

Каждое дерево имеет variance $v=9$. Корреляция прогнозов любых двух разных деревьев равна $\rho=0.2$.

$$
\operatorname{Var}(\bar f)
=
v\left(\rho+\frac{1-\rho}{B}\right).
$$

1. Выведите формулу через ковариации.
2. Вычислите variance при $B\in\{1,10,100\}$.
3. Найдите предел при $B\to\infty$.
4. Почему случайный выбор признаков в Random Forest помогает ансамблю?




